In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
model.save_pretrained("/content/drive/MyDrive/idiom-normal")
tokenizer.save_pretrained("/content/drive/MyDrive/idiom-normal")

('/content/drive/MyDrive/idiom-normal/tokenizer_config.json',
 '/content/drive/MyDrive/idiom-normal/tokenizer.json')

In [2]:
!pip uninstall -y peft
!pip install peft==0.10.0
!pip install transformers datasets accelerate evaluate sacrebleu

Found existing installation: peft 0.10.0
Uninstalling peft-0.10.0:
  Successfully uninstalled peft-0.10.0
  Using cached peft-0.10.0-py3-none-any.whl.metadata (13 kB)
Using cached peft-0.10.0-py3-none-any.whl (199 kB)


In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, TrainingArguments, Trainer
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
import evaluate
import pandas as pd

In [5]:
dataset = load_dataset(
    "csv",
    data_files="/content/idioms-normalV1.csv"
)

dataset = dataset["train"]

# shuffle + split
dataset = dataset.shuffle(seed=42)
dataset = dataset.train_test_split(test_size=0.2)

In [6]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

tokenizer.src_lang = "doi_Deva"
tokenizer.tgt_lang = "eng_Latn"


model.config.use_cache = False
model.gradient_checkpointing_enable()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [7]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_2_SEQ_LM"
)

model = get_peft_model(model, lora_config)


model.print_trainable_parameters()

trainable params: 2,359,296 || all params: 1,404,497,920 || trainable%: 0.16798145204800302


In [37]:
def preprocess(example):
    # This tokenizes the 'dogri' column
    model_inputs = tokenizer(
        example["dogri"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    # This tokenizes the 'english' column
    labels = tokenizer(
        text_target=example["english"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Apply it and kill the extra columns
tokenized_dataset = dataset.map(
    preprocess,
    batched=True,
    remove_columns=dataset["train"].column_names
)

Map:   0%|          | 0/172 [00:00<?, ? examples/s]

Map:   0%|          | 0/43 [00:00<?, ? examples/s]

In [38]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

In [39]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=30,
    save_strategy="no",
    logging_steps=10
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"]
)

trainer.train()

Step,Training Loss
10,14.034802
20,13.794228
30,13.629854
40,13.297948
50,13.098993
60,12.889886
70,12.755675
80,12.575761
90,12.410298
100,12.278336


TrainOutput(global_step=1290, training_loss=8.779900679477425, metrics={'train_runtime': 752.3319, 'train_samples_per_second': 6.859, 'train_steps_per_second': 1.715, 'total_flos': 2446813235773440.0, 'train_loss': 8.779900679477425, 'epoch': 30.0})

In [57]:
def translate(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True).to("cuda")
    forced_token = tokenizer.convert_tokens_to_ids("eng_Latn")


    outputs = model.generate(
        **inputs,
        forced_bos_token_id=forced_token,
        max_length=128,
        num_beams=2,
        repetition_penalty=2.5,
        no_repeat_ngram_size=2,
        length_penalty=1.0
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [58]:
predictions = []
references = []
types = []

for example in dataset["test"]:
    pred = translate(example["dogri"])

    predictions.append(pred)
    references.append(example["english"])
    types.append(example["type"])

In [59]:
idiom_preds, idiom_refs = [], []
normal_preds, normal_refs = [], []

for i in range(len(predictions)):
    if types[i] == "idiom":
        idiom_preds.append(predictions[i])
        idiom_refs.append(references[i])
    else:
        normal_preds.append(predictions[i])
        normal_refs.append(references[i])

In [60]:
meteor = evaluate.load("meteor")

idiom_score = meteor.compute(predictions=idiom_preds, references=idiom_refs)
normal_score = meteor.compute(predictions=normal_preds, references=normal_refs)

print("Idiom METEOR:", idiom_score["meteor"])
print("Normal METEOR:", normal_score["meteor"])

Idiom METEOR: 0.005208333333333333
Normal METEOR: 0.019892393837666677


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [61]:
df = pd.DataFrame({
    "dogri": [ex["dogri"] for ex in dataset["test"]],
    "Prediction": predictions,
    "Reference": references,
    "Type": types
})

df.to_csv("/content/final_results.csv", index=False)

In [62]:
print(f"Dogri: {dataset['test'][0]['dogri']}")
print(f"Prediction: {predictions[0]}")
print(f"Reference: {references[0]}")

Dogri: क्या तुस मीगी सुनी सकदे ओ?
Prediction: can cancanandiandiundaunda cont cont
Reference: can you hear me?
